<a href="https://colab.research.google.com/github/himanshusar123/-ClaimVision-AI/blob/main/Day_27_EY.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip install -U agent-framework-foundry azure-identity

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 872.0/872.0 kB 8.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 552.9/552.9 kB 8.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 17.6 MB/s  0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 2.54.0
    Uninstalling openai-2.54.0:
      Successfully uninstalled openai-2.54.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13/13 [agent-framework-foundry]


In [1]:
from agent_framework import Agent
from agent_framework.foundry import FoundryChatClient
from azure.identity import DeviceCodeCredential

print("All imports successful")

All imports successful


In [2]:
FOUNDRY_PROJECT_ENDPOINT = input(
    "Enter your Microsoft Foundry project endpoint: "
).strip()

FOUNDRY_MODEL = input(
    "Enter the exact deployed model name: "
).strip()

assert FOUNDRY_PROJECT_ENDPOINT.startswith("https://"), (
    "Please enter a valid HTTPS project endpoint."
)
assert FOUNDRY_MODEL, "The deployment name cannot be empty."

print("Endpoint configured.")
print("Deployment name:", FOUNDRY_MODEL)

Enter your Microsoft Foundry project endpoint: https://sb-mazeazusrqn9t23-bp41-resource.services.ai.azure.com/api/projects/sb-mazeazusrqn9t23-bp41-880412
Enter the exact deployed model name: gpt-6.1-sol
Endpoint configured.
Deployment name: gpt-6.1-sol


In [3]:
credential = DeviceCodeCredential()

print("Authentication credential created.")

Authentication credential created.


In [4]:
client = FoundryChatClient(
    project_endpoint=FOUNDRY_PROJECT_ENDPOINT,
    model=FOUNDRY_MODEL,
    credential=credential,
)

print("Foundry client initialized.")

Foundry client initialized.


In [5]:
test_agent = Agent(
    client=client,
    name="ConnectionTest",
    instructions="You are a helpful assistant. Keep answers brief.",
)

response = await test_agent.run(
    "Reply with: Azure AI Foundry connection successful."
)

print(response)

To sign in, use a web browser to open the page https://login.microsoft.com/device and enter the code AMXCQP9DT to authenticate.
Azure AI Foundry connection successful.


In [6]:
investigator = Agent(
    client=client,
    name="OrderInvestigator",
    instructions=(
        "You are an order escalation analyst. Use only facts supplied "
        "in the prompt or returned by tools. Never invent delivery dates, "
        "courier updates, refunds, or approvals. Separate verified facts, "
        "unknowns, and recommended next steps. Do not perform external actions."
    ),
)

print("Order Investigator agent created.")

Order Investigator agent created.


In [8]:
ORDER_SNAPSHOT = {
    "order_id": "ORD78421",
    "customer_id": "CUS2048",
    "customer_name": "Neha Verma",
    "product": "NovaBook Pro laptop",
    "order_value_inr": 89999,

    "payment_status": "Paid",
    "order_status": "In Transit",

    "promised_delivery": "13 Nov 2026",

    "warehouse_id": "WH-NCR-02",
    "inventory_quantity": 0,
    "inventory_last_sync": "12 Nov 2026 18:00",

    "courier_event_time": "13 Nov 2026 07:40",
    "courier_event": "Address verification required",

    "latest_scan_known": False,

    "escalation_id": "ESC-3107",
    "priority": "High"
}

print("Order snapshot created successfully!")

print(ORDER_SNAPSHOT)

Order snapshot created successfully!
{'order_id': 'ORD78421', 'customer_id': 'CUS2048', 'customer_name': 'Neha Verma', 'product': 'NovaBook Pro laptop', 'order_value_inr': 89999, 'payment_status': 'Paid', 'order_status': 'In Transit', 'promised_delivery': '13 Nov 2026', 'warehouse_id': 'WH-NCR-02', 'inventory_quantity': 0, 'inventory_last_sync': '12 Nov 2026 18:00', 'courier_event_time': '13 Nov 2026 07:40', 'courier_event': 'Address verification required', 'latest_scan_known': False, 'escalation_id': 'ESC-3107', 'priority': 'High'}


In [9]:
print("Order ID:", ORDER_SNAPSHOT["order_id"])
print("Customer:", ORDER_SNAPSHOT["customer_name"])
print("Order Status:", ORDER_SNAPSHOT["order_status"])
print("Priority:", ORDER_SNAPSHOT["priority"])

Order ID: ORD78421
Customer: Neha Verma
Order Status: In Transit
Priority: High


In [10]:
prompt = f"""
Investigate this escalation using only this snapshot:

{ORDER_SNAPSHOT}

Return:
1. Verified facts
2. Unknown or stale information
3. Recommended next step
4. Customer-facing statement that must be avoided
"""

answer = await investigator.run(prompt)
print(answer)

## 1. Verified facts
- **Escalation:** ESC-3107, marked **High** priority.
- **Order:** ORD78421 for Neha Verma (CUS2048), containing a NovaBook Pro laptop.
- **Order value:** ₹89,999; payment status is **Paid**.
- **Recorded order status:** In Transit.
- **Promised delivery date:** 13 Nov 2026.
- **Recorded courier event:** “Address verification required,” timestamped 13 Nov 2026 at 07:40.
- **Warehouse:** WH-NCR-02; recorded inventory quantity is **0**, last synced on 12 Nov 2026 at 18:00.
- The snapshot explicitly indicates that the **latest scan is not known**.

## 2. Unknown or stale information
- The shipment’s current location, latest courier scan, and whether the address-verification issue has been resolved are unknown.
- No updated delivery estimate or proof of delivery is supplied. The promised date is not confirmation of delivery.
- No current date/time is supplied, so whether the promised date has passed cannot be determined from this snapshot alone.
- Inventory is a last-s

In [11]:
def lookup_order(order_id: str) -> dict:
    """Look up an order in the local training snapshot."""

    if order_id != ORDER_SNAPSHOT["order_id"]:
        return {
            "found": False,
            "message": "No matching order in the training snapshot."
        }

    return {
        "found": True,
        "data": ORDER_SNAPSHOT.copy()
    }


# Test the tool
lookup_order("ORD78421")

{'found': True,
 'data': {'order_id': 'ORD78421',
  'customer_id': 'CUS2048',
  'customer_name': 'Neha Verma',
  'product': 'NovaBook Pro laptop',
  'order_value_inr': 89999,
  'payment_status': 'Paid',
  'order_status': 'In Transit',
  'promised_delivery': '13 Nov 2026',
  'warehouse_id': 'WH-NCR-02',
  'inventory_quantity': 0,
  'inventory_last_sync': '12 Nov 2026 18:00',
  'courier_event_time': '13 Nov 2026 07:40',
  'courier_event': 'Address verification required',
  'latest_scan_known': False,
  'escalation_id': 'ESC-3107',
  'priority': 'High'}}

In [12]:
lookup_order("ORD00000")

{'found': False, 'message': 'No matching order in the training snapshot.'}

In [13]:
PROHIBITED_ACTIONS = {
    "refund",
    "cancel_order",
    "change_address",
    "change_shipment",
    "contact_courier",
    "send_message"
}


def validate_recommendation(text: str) -> dict:
    """Flag unsupported delivery promises and actions needing review."""

    lowered = text.lower()
    warnings = []

    if (
        "will arrive on" in lowered
        or "guaranteed delivery" in lowered
    ):
        warnings.append(
            "Unsupported delivery promise detected."
        )

    if any(
        action.replace("_", " ") in lowered
        for action in PROHIBITED_ACTIONS
    ):
        warnings.append(
            "Potential consequential action mentioned; "
            "human review required."
        )

    return {
        "approved_for_automatic_execution": False,
        "warnings": warnings,
        "requires_human_review": True
    }


# Test the validator
validate_recommendation(
    "The order is in transit; the revised delivery date is not available."
)

{'approved_for_automatic_execution': False,
 'warnings': [],
 'requires_human_review': True}

In [14]:
validate_recommendation(
    "The order will arrive on 14 Nov 2026. Send message to the customer."
)

{'approved_for_automatic_execution': False,
 'warnings': ['Unsupported delivery promise detected.',
  'Potential consequential action mentioned; human review required.'],
 'requires_human_review': True}

In [15]:
from agent_framework import Agent, tool


@tool
def get_order_snapshot(order_id: str) -> str:
    """Return the read-only training snapshot for an order ID."""

    result = lookup_order(order_id)

    if not result["found"]:
        return result["message"]

    return str(result["data"])

In [16]:
investigator = Agent(
    client=client,
    name="OrderInvestigator",
    instructions=(
        "You are an order escalation analyst. "
        "Use only facts supplied in the prompt or returned by tools. "
        "Never invent delivery dates, courier updates, refunds, or approvals. "
        "Separate verified facts, unknowns, and recommended next steps. "
        "Do not perform external actions. "
        "Use get_order_snapshot when asked to investigate an order."
    ),
    tools=[get_order_snapshot],
)

In [17]:
answer = await investigator.run(
    "Investigate order ORD78421 using the order lookup tool. "
    "Report verified facts, unknowns, recommended next step, "
    "and one customer-facing claim that must be avoided."
)

print(answer)

## Order ORD78421

### Verified facts
- **Customer:** Neha Verma (CUS2048).
- **Product:** NovaBook Pro laptop; order value **₹89,999**.
- **Payment:** Paid. **Order status:** In Transit.
- **Recorded promised delivery date:** 13 Nov 2026.
- **Recorded courier event:** “Address verification required,” timestamped 13 Nov 2026 at 07:40.
- **Latest scan known:** No.
- **Warehouse:** WH-NCR-02; recorded inventory quantity is **0**, last synced 12 Nov 2026 at 18:00.
- **Escalation:** ESC-3107, marked **High** priority.

### Unknowns
- Current parcel location and latest courier scan.
- What address information needs verification and whether it has been resolved.
- Whether the recorded delivery promise remains achievable; no revised delivery date is supplied.
- Whether the warehouse inventory record affects this already in-transit order.

### Recommended next step
Through high-priority escalation **ESC-3107**, request clarification of the address-verification requirement and obtain the latest

In [18]:
answer = await investigator.run(
    "Investigate order ORD78421 using the order lookup tool. "
    "Report verified facts, unknowns, recommended next step, "
    "and one customer-facing claim that must be avoided."
)

print(answer)

### Verified facts
- **Order:** ORD78421 — Neha Verma’s NovaBook Pro laptop.
- **Value/payment:** ₹89,999; payment marked **Paid**.
- **Order status:** **In Transit**.
- **Recorded promised delivery:** 13 Nov 2026—not confirmation of actual delivery.
- **Courier event:** “Address verification required,” recorded on 13 Nov 2026 at 07:40.
- **Latest scan:** Not known in the snapshot.
- **Warehouse:** WH-NCR-02; inventory recorded as **0**, last synced on 12 Nov 2026 at 18:00.
- **Escalation:** ESC-3107, marked **High** priority.

### Unknowns
- Current parcel location and any subsequent courier updates.
- Which address details require verification and whether verification has been completed.
- Whether the promised delivery was met, and any confirmed revised delivery estimate.
- Whether a replacement is available or any refund/replacement has been approved.

### Recommended next step
Through high-priority escalation **ESC-3107**, request clarification of the address-verification requireme

In [19]:
session = investigator.create_session()

first_response = await investigator.run(
    "Look up order ORD78421 and summarize its current status.",
    session=session
)

print(first_response)

### Order ORD78421 — Current status

**Verified facts**
- **Product:** NovaBook Pro laptop
- **Payment:** Paid — ₹89,999
- **Order status:** In Transit
- **Promised delivery:** 13 Nov 2026
- **Recorded courier event:** “Address verification required,” dated 13 Nov 2026 at 07:40
- **Escalation:** ESC-3107, marked **High** priority

**Unknowns**
- The latest courier scan and current package location are not known.
- Delivery completion or a revised delivery date is not confirmed.

**Recommended next steps**
- Verify the delivery address with the customer.
- Request an updated courier scan and delivery estimate through the existing escalation.

No external actions have been taken.


In [20]:
follow_up = await investigator.run(
    "What information is still unknown?",
    session=session
)

print(follow_up)

The following information is still **unknown or unconfirmed** for order ORD78421:

- **Latest courier scan and current package location:** The snapshot explicitly says the latest scan is not known.
- **Address-verification details:** What needs verification and whether it has been resolved.
- **Delivery outcome:** Whether the order was delivered or the promised date of 13 Nov 2026 was met.
- **Updated delivery estimate:** No revised date is provided.
- **Escalation progress:** ESC-3107 is marked High priority, but no resolution or further update is provided.


In [21]:
# Create a separate session for a different case.
another_case_session = investigator.create_session()

unknown_response = await investigator.run(
    "Investigate order ORD00000. Do not assume it exists.",
    session=another_case_session
)

print(unknown_response)

### Verified facts
- The lookup for **ORD00000** returned **no matching order in the training snapshot**.

### Unknowns
- Whether this order exists outside the snapshot.
- Its status, delivery details, and any escalation history.

### Recommended next steps
- Confirm that the order ID is correct.
- If correct, request verification in the live order system through an authorized support channel.

No external actions were taken.


In [22]:
def investigate_snapshot(snapshot: dict) -> dict:
    """Analyze only the facts available in the supplied snapshot."""

    return {
        "verified_facts": [
            f"Order {snapshot['order_id']} is "
            f"{snapshot['order_status']}.",
            f"Payment status is {snapshot['payment_status']}.",
            f"Courier event: {snapshot['courier_event']} "
            f"at {snapshot['courier_event_time']}.",
            f"Promised delivery date in snapshot: "
            f"{snapshot['promised_delivery']}.",
            f"Warehouse inventory snapshot quantity: "
            f"{snapshot['inventory_quantity']}."
        ],
        "unknowns": [
            "No newer courier scan is present in the supplied snapshot.",
            "No revised delivery estimate is available.",
            "The inventory snapshot may be stale.",
            "The current outcome of the address-verification process "
            "is not supplied."
        ],
        "recommendation": (
            "A support representative should verify the latest courier "
            "status and address-verification process through approved "
            "internal channels before updating the customer."
        ),
        "customer_claim_to_avoid": (
            "Do not promise a new delivery date or claim the courier "
            "has been contacted."
        )
    }


analysis_result = investigate_snapshot(ORDER_SNAPSHOT)
analysis_result

{'verified_facts': ['Order ORD78421 is In Transit.',
  'Payment status is Paid.',
  'Courier event: Address verification required at 13 Nov 2026 07:40.',
  'Promised delivery date in snapshot: 13 Nov 2026.',
  'Warehouse inventory snapshot quantity: 0.'],
 'unknowns': ['No newer courier scan is present in the supplied snapshot.',
  'No revised delivery estimate is available.',
  'The inventory snapshot may be stale.',
  'The current outcome of the address-verification process is not supplied.'],
 'recommendation': 'A support representative should verify the latest courier status and address-verification process through approved internal channels before updating the customer.',
 'customer_claim_to_avoid': 'Do not promise a new delivery date or claim the courier has been contacted.'}

In [24]:
print("run_workflow exists:", "run_workflow" in globals())
print("lookup_order exists:", "lookup_order" in globals())
print("investigate_snapshot exists:", "investigate_snapshot" in globals())

run_workflow exists: False
lookup_order exists: True
investigate_snapshot exists: True


In [25]:
def run_workflow(
    order_id: str,
    human_approval: str = "PENDING"
) -> dict:

    decision = human_approval.upper()

    # Validate approval decision
    if decision not in {"PENDING", "APPROVED", "REJECTED"}:
        return {
            "status": "STOPPED",
            "reason": "Invalid approval decision.",
            "external_action_taken": False
        }

    # Step 1: Order lookup
    lookup = lookup_order(order_id)

    if not lookup["found"]:
        return {
            "status": "STOPPED",
            "reason": lookup["message"],
            "external_action_taken": False
        }

    # Step 2: Investigation
    analysis = investigate_snapshot(lookup["data"])

    # Step 3: Policy validation
    validation = validate_recommendation(
        analysis["recommendation"]
    )

    # Step 4: Human approval
    if decision == "PENDING":
        status = "WAITING_FOR_HUMAN_REVIEW"

    elif decision == "REJECTED":
        status = "REJECTED_BY_HUMAN_REVIEW"

    else:
        status = "APPROVED_FOR_MANUAL_FOLLOW_UP"

    # No external action is performed
    return {
        "status": status,
        "case_id": lookup["data"]["escalation_id"],
        "order_id": lookup["data"]["order_id"],
        "analysis": analysis,
        "policy_check": validation,
        "human_decision": decision,
        "external_action_taken": False
    }


print("run_workflow() defined successfully!")

run_workflow() defined successfully!


In [26]:
pending = run_workflow(
    "ORD78421",
    human_approval="PENDING"
)

print(pending)

{'status': 'WAITING_FOR_HUMAN_REVIEW', 'case_id': 'ESC-3107', 'order_id': 'ORD78421', 'analysis': {'verified_facts': ['Order ORD78421 is In Transit.', 'Payment status is Paid.', 'Courier event: Address verification required at 13 Nov 2026 07:40.', 'Promised delivery date in snapshot: 13 Nov 2026.', 'Warehouse inventory snapshot quantity: 0.'], 'unknowns': ['No newer courier scan is present in the supplied snapshot.', 'No revised delivery estimate is available.', 'The inventory snapshot may be stale.', 'The current outcome of the address-verification process is not supplied.'], 'recommendation': 'A support representative should verify the latest courier status and address-verification process through approved internal channels before updating the customer.', 'customer_claim_to_avoid': 'Do not promise a new delivery date or claim the courier has been contacted.'}, 'policy_check': {'approved_for_automatic_execution': False, 'warnings': [], 'requires_human_review': True}, 'human_decision':

In [27]:
pending = run_workflow(
    "ORD78421",
    human_approval="PENDING"
)

approved = run_workflow(
    "ORD78421",
    human_approval="APPROVED"
)

rejected = run_workflow(
    "ORD78421",
    human_approval="REJECTED"
)

unknown = run_workflow("ORD00000")

print("Pending:", pending["status"])
print("Approved:", approved["status"])
print("Rejected:", rejected["status"])
print("Unknown order:", unknown["status"])

print(
    "External action after approval:",
    approved["external_action_taken"]
)

Pending: WAITING_FOR_HUMAN_REVIEW
Approved: APPROVED_FOR_MANUAL_FOLLOW_UP
Rejected: REJECTED_BY_HUMAN_REVIEW
Unknown order: STOPPED
External action after approval: False


In [28]:
def human_review_prompt(report: dict) -> None:

    print("=" * 55)
    print("HUMAN APPROVAL CHECKPOINT")
    print("=" * 55)

    print("Case ID:", report.get("case_id"))
    print("Order ID:", report.get("order_id"))
    print("Current Status:", report.get("status"))

    print("\nRECOMMENDATION:")
    print(
        report.get("analysis", {}).get(
            "recommendation"
        )
    )

    print("\nCUSTOMER CLAIM TO AVOID:")
    print(
        report.get("analysis", {}).get(
            "customer_claim_to_avoid"
        )
    )

    print("\nPOLICY WARNINGS:")
    print(
        report.get("policy_check", {}).get("warnings")
    )

    print("\nEXTERNAL ACTION TAKEN: NO")
    print("Reviewer decision required.")

In [29]:
human_review_prompt(pending)

HUMAN APPROVAL CHECKPOINT
Case ID: ESC-3107
Order ID: ORD78421
Current Status: WAITING_FOR_HUMAN_REVIEW

RECOMMENDATION:
A support representative should verify the latest courier status and address-verification process through approved internal channels before updating the customer.

CUSTOMER CLAIM TO AVOID:
Do not promise a new delivery date or claim the courier has been contacted.

POLICY WARNINGS:
[]

EXTERNAL ACTION TAKEN: NO
Reviewer decision required.


In [30]:
human_review_prompt(rejected)

HUMAN APPROVAL CHECKPOINT
Case ID: ESC-3107
Order ID: ORD78421
Current Status: REJECTED_BY_HUMAN_REVIEW

RECOMMENDATION:
A support representative should verify the latest courier status and address-verification process through approved internal channels before updating the customer.

CUSTOMER CLAIM TO AVOID:
Do not promise a new delivery date or claim the courier has been contacted.

POLICY WARNINGS:
[]

EXTERNAL ACTION TAKEN: NO
Reviewer decision required.


In [31]:
reflection = {
    "Deterministic_tasks": [
        "Order lookup",
        "Payment status verification",
        "Policy validation",
        "Approval gate"
    ],

    "Agent_tasks": [
        "Investigating order escalation",
        "Summarizing verified facts",
        "Identifying unknown information",
        "Recommending next steps"
    ],

    "Audit_requirements": [
        "Order ID and escalation ID",
        "Tool execution logs",
        "Verified source data",
        "Human approver identity",
        "Approval or rejection decision",
        "Timestamp of decision"
    ]
}

for section, items in reflection.items():
    print("\n" + section.replace("_", " ").upper())

    for item in items:
        print("-", item)


DETERMINISTIC TASKS
- Order lookup
- Payment status verification
- Policy validation
- Approval gate

AGENT TASKS
- Investigating order escalation
- Summarizing verified facts
- Identifying unknown information
- Recommending next steps

AUDIT REQUIREMENTS
- Order ID and escalation ID
- Tool execution logs
- Verified source data
- Human approver identity
- Approval or rejection decision
- Timestamp of decision
